# Build Multi-Agent Collaboration with Strands + Amazon Bedrock AgentCore

This notebook walks you through deploying the four agents in the mortgage
customer service multi-agent system. Each agent is:

1. Defined as a Python file with a Strands `Agent` wrapped in a `@app.entrypoint`
2. Packaged and deployed to **Amazon Bedrock AgentCore Runtime**
3. Given an endpoint (ARN) that the supervisor agent calls as a tool

**Architecture**
```
Customer email
    └─► Supervisor Agent         (AgentCore Runtime — Nova Pro)
              ├─► Intent Classifier Agent   (AgentCore Runtime — Nova Lite 2)
              ├─► Tech Specialist Agent     (AgentCore Runtime — Nova Lite 2)
              │         ├─► EarlyCheckErrorCheckFunction  (Lambda)
              │         └─► DUSystemStatusCheckFunction   (Lambda)
              └─► Policy Specialist Agent  (AgentCore Runtime — Nova Lite 2)
                        ├─► PolicySellingGuideKB  (Bedrock Knowledge Base)
                        └─► CaseLookupFunction    (Lambda)
```

## Task 2.1 — Setup

In [ ]:
import boto3
import json
import zipfile
import os
import shutil
import subprocess
import time

REGION     = "us-east-1"
ACCOUNT_ID = boto3.client("sts").get_caller_identity()["Account"]

agentcore_control = boto3.client("bedrock-agentcore-control", region_name=REGION)
agentcore_runtime = boto3.client("bedrock-agentcore",         region_name=REGION)
cfn_client        = boto3.client("cloudformation",            region_name=REGION)

print(f"Account: {ACCOUNT_ID}")

### Paste your lab resource IDs below

Look at the **left panel** of the lab interface (where you found the `LabInstanceURL`).
You will see three output values:

| Output | Variable |
|---|---|
| `AgentCoreExecutionRoleArn` | `EXECUTION_ROLE_ARN` |
| `KnowledgeBaseId` | `KNOWLEDGE_BASE_ID` |
| `DeployBucketName` | `S3_BUCKET` |

Copy each value and paste it between the quotes in the cell below, then run the cell.

In [ ]:
EXECUTION_ROLE_ARN = ""  # Paste AgentCoreExecutionRoleArn here
KNOWLEDGE_BASE_ID  = ""  # Paste KnowledgeBaseId here
S3_BUCKET          = ""  # Paste DeployBucketName here

print(f"Execution Role : {EXECUTION_ROLE_ARN}")
print(f"Knowledge Base : {KNOWLEDGE_BASE_ID}")
print(f"Deploy Bucket  : {S3_BUCKET}")


## Helper: package and deploy an agent to AgentCore Runtime

Run this cell once. `deploy_agent` and `invoke_agent` are used throughout the lab — you do not need to modify this code.

In [5]:
def deploy_agent(agent_name: str, agent_file: str, env_vars: dict = None) -> str:
    """
    Packages agent_file + dependencies, uploads to S3, and creates
    an AgentCore Runtime. Returns the agentRuntimeArn.
    """
    print(f"\n--- Deploying {agent_name} ---")
    zip_path = f"/tmp/{agent_name}.zip"
    pkg_dir  = f"/tmp/{agent_name}_pkg"

    # Use pre-staged arm64 dependencies (installed by the lab bootstrap
    # script into /opt/agentcore_deps or /tmp/agentcore_deps). If neither
    # path exists — e.g. install.sh failed — fall back to a per-session
    # on-demand install so the lab still works.
    print("Installing dependencies for arm64...")
    _STAGED = "/opt/agentcore_deps"
    _CACHE  = "/tmp/agentcore_deps"
    def _has_content(p): return os.path.isdir(p) and bool(os.listdir(p))
    _deps_src = _STAGED if _has_content(_STAGED) else _CACHE
    if not _has_content(_deps_src):
        print("  Pre-staged deps not found -- installing now (one-time, ~60 s)...")
        subprocess.check_call([
            "uv", "pip", "install",
            "--python-platform", "aarch64-manylinux2014",
            "--python-version", "3.13",
            f"--target={_CACHE}",
            "--only-binary=:all:",
            "bedrock-agentcore", "strands-agents", "boto3",
        ])
        _deps_src = _CACHE
    shutil.rmtree(pkg_dir, ignore_errors=True)
    shutil.copytree(_deps_src, pkg_dir)

    # Read the agent source and prepend env var injection
    agent_src = open(f"agents/{agent_file}").read()
    if env_vars:
        env_block = "\n".join(
            f"os.environ.setdefault('{k}', '{v}')" for k, v in env_vars.items()
        )
        agent_src = f"import os\n{env_block}\n\n" + agent_src

    # Write as entrypoint.py (AgentCore requires this filename)
    with open(f"{pkg_dir}/entrypoint.py", "w") as f:
        f.write(agent_src)

    # Zip everything
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for root, _, files in os.walk(pkg_dir):
            for file in files:
                file_path = os.path.join(root, file)
                zf.write(file_path, os.path.relpath(file_path, pkg_dir))

    # Upload to S3
    s3_key = f"{agent_name}/deployment_package.zip"
    print(f"Uploading to s3://{S3_BUCKET}/{s3_key}")
    boto3.client("s3", region_name=REGION).upload_file(
        zip_path, S3_BUCKET, s3_key,
        ExtraArgs={"ExpectedBucketOwner": ACCOUNT_ID}
    )

    # Create AgentCore Runtime
    print("Creating AgentCore Runtime...")
    resp = agentcore_control.create_agent_runtime(
        agentRuntimeName=agent_name,
        agentRuntimeArtifact={
            "codeConfiguration": {
                "code": {
                    "s3": {"bucket": S3_BUCKET, "prefix": s3_key}
                },
                "runtime": "PYTHON_3_13",
                "entryPoint": ["entrypoint.py"],
            }
        },
        networkConfiguration={"networkMode": "PUBLIC"},
        roleArn=EXECUTION_ROLE_ARN,
        lifecycleConfiguration={
            "idleRuntimeSessionTimeout": 300,  # 5 min
            "maxLifetime": 1800,               # 30 min
        },
    )

    arn = resp["agentRuntimeArn"]
    print(f"Created: {arn}  (status: {resp['status']})")

    # Wait for the runtime to become READY
    print("Waiting for runtime to become READY...")
    runtime_id = arn.split("/")[-1]
    for _ in range(30):  # up to 5 minutes
        status = agentcore_control.get_agent_runtime(agentRuntimeId=runtime_id)["status"]
        if status == "READY":
            print(f"Runtime is READY.")
            break
        print(f"  Status: {status} -- waiting...")
        time.sleep(10)
    else:
        print("WARNING: Runtime did not reach READY state within 5 minutes.")

    return arn


def invoke_agent(agent_arn: str, prompt: str) -> str:
    """Quick test helper -- invoke a deployed AgentCore Runtime.
    Retries once if the runtime is still cold-starting (initialization timeout).
    """
    import uuid
    from botocore.exceptions import ClientError
    for attempt in range(2):
        try:
            resp = agentcore_runtime.invoke_agent_runtime(
                agentRuntimeArn=agent_arn,
                qualifier="DEFAULT",
                runtimeSessionId=str(uuid.uuid4()),
                payload=json.dumps({"prompt": prompt}),
            )
            raw = resp["response"].read()
            return json.loads(raw).get("result", str(raw))
        except ClientError as e:
            if attempt == 0 and "initialization time exceeded" in e.response.get("Error", {}).get("Message", ""):
                print("  Runtime is still initializing -- retrying in 30 s...")
                time.sleep(30)
            else:
                raise

### Troubleshooting: deployed with incomplete code?

If you ran a deploy cell before completing the `.py` file, the runtime was created with broken code.
**You do not need to restart the lab.** Run `delete_runtime("agent_name")` below to remove it,
then complete the `.py` file and re-run the deploy cell.

In [6]:
def delete_runtime(agent_name):
    """Delete a runtime by name. Use this if you need to redeploy after fixing a .py file."""
    runtimes = agentcore_control.list_agent_runtimes()["agentRuntimes"]
    match = [r for r in runtimes if r["agentRuntimeName"] == agent_name]
    if not match:
        print(f"No runtime found with name '{agent_name}'")
        return
    agentcore_control.delete_agent_runtime(agentRuntimeId=match[0]["agentRuntimeId"])
    print(f"Deleted runtime: {agent_name}")
    print("You can now fix the .py file and re-run the deploy cell.")

# Example usage -- uncomment and run if needed:
# delete_runtime("intent_classification_agent")
# delete_runtime("tech_specialist_agent")
# delete_runtime("policy_specialist_agent")
# delete_runtime("customer_service_supervisor")


## Task 2.2 — Deploy the Intent Classification Agent

> **📋 Switch to the instructions tab** and complete **Task 2.2** before running the deploy cell below.
> Open `agents/intent_classifier.py`, fill in the system prompt, and create the agent.
>
> **⚠️ Do not run the deploy cell until the agent file is complete.** If you do, use `delete_runtime("agent_name")` from the helper cell above to remove it, then fix the file and redeploy.

In [ ]:
INTENT_CLASSIFIER_ARN = deploy_agent(
    agent_name="intent_classification_agent",
    agent_file="intent_classifier.py",
)
print(f"\nIntent Classifier ARN: {INTENT_CLASSIFIER_ARN}")

In [8]:
# Test the Intent Classification Agent
result = invoke_agent(
    INTENT_CLASSIFIER_ARN,
    "I'm having trouble accessing Desktop Underwriter. I keep getting Connection failed."
)
print(result)
# Expected: {"category": "System & Technology Issues", "subcategory": "Desktop Underwriter (DU) access problems"}

```json
{
    "category": "System & Technology Issues",
    "subcategory": "Desktop Underwriter (DU) access problems"
}
```



## Task 2.3 — Deploy the Tech Specialist Agent

> **📋 Switch to the instructions tab** and complete **Task 2.3** before running the deploy cell below.
> Open `agents/tech_specialist.py`, add the two tool functions, define the system prompt, and create the agent.
>
> **⚠️ Do not run the deploy cell until the agent file is complete.** If you do, use `delete_runtime("agent_name")` from the helper cell above to remove it, then fix the file and redeploy.

In [ ]:
TECH_SPECIALIST_ARN = deploy_agent(
    agent_name="tech_specialist_agent",
    agent_file="tech_specialist.py",
)
print(f"\nTech Specialist ARN: {TECH_SPECIALIST_ARN}")

In [13]:
# Test the Tech Specialist Agent
result = invoke_agent(
    TECH_SPECIALIST_ARN,
    "I'm getting error code EC-405 in EarlyCheck when submitting a loan. What does this mean?"
)
print(result)

I apologize for the technical issue you're experiencing. It looks like there's an AWS permissions error occurring when trying to look up the EC-405 error code. This appears to be a system configuration issue rather than something you can resolve directly.

However, I can tell you that **EC-405 in EarlyCheck** typically indicates:

**"Income Verification Failed"** - This error means EarlyCheck was unable to verify the borrower's income documentation. Common causes include:

1. Missing or incomplete income documentation
2. Discrepancies between submitted documents and application information
3. Documents that can't be automatically processed (e.g., non-standard pay stubs)
4. Expired or outdated documentation

## Recommended Troubleshooting Steps:

1. **Review Submitted Documents**: 
   - Verify all income documents (pay stubs, W-2s, tax returns) are attached and complete
   - Ensure documents match the employment information in the application

2. **Check Document Quality**:
   - Make su

## Task 2.4 — Verify the Knowledge Base

> Run this cell to confirm the pre-created knowledge base is ready. Status must be `ACTIVE` before proceeding to Task 2.5.

In [12]:
bedrock_agent_client = boto3.client("bedrock-agent", region_name=REGION)
kb_detail = bedrock_agent_client.get_knowledge_base(knowledgeBaseId=KNOWLEDGE_BASE_ID)
print(f"Name  : {kb_detail['knowledgeBase']['name']}")
print(f"Status: {kb_detail['knowledgeBase']['status']}")
# Expected: Status = ACTIVE

Name  : PolicySellingGuideKB
Status: ACTIVE


## Task 2.5 — Deploy the Policy Specialist Agent

> **📋 Switch to the instructions tab** and complete **Task 2.5** before running the deploy cell below.
> Open `agents/policy_specialist.py`, add the two tool functions, define the system prompt, and create the agent.
>
> **⚠️ Do not run the deploy cell until the agent file is complete.** If you do, use `delete_runtime("agent_name")` from the helper cell above to remove it, then fix the file and redeploy.

In [ ]:
POLICY_SPECIALIST_ARN = deploy_agent(
    agent_name="policy_specialist_agent",
    agent_file="policy_specialist.py",
    env_vars={"KNOWLEDGE_BASE_ID": KNOWLEDGE_BASE_ID},
)
print(f"\nPolicy Specialist ARN: {POLICY_SPECIALIST_ARN}")

In [15]:
# Test the Policy Specialist Agent
result = invoke_agent(
    POLICY_SPECIALIST_ARN,
    "What are the requirements for HomeReady mortgage programs regarding homebuyer education?"
)
print(result)

The HomeReady Mortgage program requires borrowers to complete homebuyer education as part of the eligibility requirements. Specifically:

**Homebuyer Education Requirements for HomeReady Mortgages:**
1. **Mandatory Counseling**: Borrowers must complete homebuyer education counseling from an approved housing counseling agency. This is a requirement for all HomeReady Mortgage applicants.

2. **Content Coverage**: The counseling must cover topics including but not limited to:
   - Homeownership responsibilities
   - Budgeting and financial management
   - Mortgage process overview
   - Home maintenance and repair basics
   - Understanding closing costs and fees

3. **Documentation**: Lenders must obtain and retain documentation of completion of the homebuyer education requirements.

4. **Effective Date**: These requirements are effective as of April 2, 2025 (as indicated in B2-2-06, Homeownership Education and Housing Counseling).

**Source Reference:**
- **B2-2-06, Homeownership Educatio

## Task 2.6 — Deploy the Supervisor Agent

> **📋 Switch to the instructions tab** and complete **Task 2.6** before running the deploy cells below.
> Open `agents/supervisor.py`, add the three specialist tool functions, define the system prompt, and create the agent.
>
> **⚠️ Do not run the deploy cells until the agent file is complete.**

In [ ]:
# Create AgentCore Memory for the supervisor (takes 1-2 minutes)
# This enables the supervisor to remember context across conversation turns.
from bedrock_agentcore.memory import MemoryClient

mem_client = MemoryClient(region_name=REGION)
memory = mem_client.create_memory_and_wait(
    name="SupervisorSessionMemory",
    description="Session memory for the customer service supervisor agent",
    strategies=[
        {"summaryMemoryStrategy": {
            "name": "SessionSummarizer",
            "namespaceTemplates": ["/summaries/{actorId}/{sessionId}/"]
        }},
        {"semanticMemoryStrategy": {
            "name": "FactExtractor",
            "namespaceTemplates": ["/facts/{actorId}/"]
        }},
    ]
)
MEMORY_ID = memory["id"]
print(f"Memory ID: {MEMORY_ID}")

In [ ]:
SUPERVISOR_ARN = deploy_agent(
    agent_name="customer_service_supervisor",
    agent_file="supervisor.py",
    env_vars={
        "INTENT_CLASSIFIER_ARN": INTENT_CLASSIFIER_ARN,
        "TECH_SPECIALIST_ARN"  : TECH_SPECIALIST_ARN,
        "POLICY_SPECIALIST_ARN": POLICY_SPECIALIST_ARN,
        "MEMORY_ID"            : MEMORY_ID,
    },
)
print(f"\nSupervisor ARN: {SUPERVISOR_ARN}")

## Task 3 — Test the Multi-Agent System

> **📋 Switch back to the instructions tab** and follow **Task 3** for the expected outputs.
> All three test cases are run through the Supervisor Agent below.

In [18]:
# Test Case 1: Technical Support Request
email_1 = """
Subject: Urgent - Desktop Underwriter Access Issues

Hi Support Team,

I'm having trouble accessing Desktop Underwriter since this morning. Every time I try to
log in, I get an error message saying \"Connection failed - unable to establish secure connection.\"
This is preventing me from processing loan applications.

I've tried clearing my browser cache, using a different browser, and restarting my computer.
Nothing seems to work.

Thanks,
Sarah Johnson
"""
print("=== Test Case 1: Technical Support ===")
print(invoke_agent(SUPERVISOR_ARN, email_1))

=== Test Case 1: Technical Support ===
<thinking> The tech_specialist has provided a detailed response to the technical issue. I will synthesize this information into a professional email reply for the customer. </thinking>

---

Subject: Re: Urgent - Desktop Underwriter Access Issues

Hi Sarah,

Thank you for reaching out and providing detailed information about the issue you're experiencing.

I apologize for the inconvenience caused by the Desktop Underwriter (DU) access problem. Our team has identified an authorization error in our backend services that is preventing the system status check from completing successfully.

Here are the immediate troubleshooting steps you can take:

1. **Check Network Connectivity:**
   - Verify your internet connection is stable.
   - Ensure you're using the correct VPN if required for DU access.
   - Try accessing other web-based applications to confirm general internet connectivity.

2. **Browser Troubleshooting:**
   - Clear browser cache and cooki

In [19]:
# Test Case 2: Policy Clarification
email_2 = """
Subject: HomeReady Program - Homebuyer Education Requirements

Hello,

I have a question about the HomeReady mortgage program requirements. I'm working with a
first-time homebuyer and need clarification on the homebuyer education requirements.

Specifically:
1. Is homebuyer education mandatory for all HomeReady borrowers?
2. What types of education courses are acceptable?
3. When does the education need to be completed -- before application or before closing?

Best regards,
Michael Chen
"""
print("=== Test Case 2: Policy Clarification ===")
print(invoke_agent(SUPERVISOR_ARN, email_2))

=== Test Case 2: Policy Clarification ===
Based on the information provided by the policy_specialist, here is the response to Michael Chen's questions about the HomeReady mortgage program requirements:

---

Hello Michael,

Thank you for your inquiry regarding the HomeReady mortgage program requirements. Here are the details regarding the homebuyer education requirements:

1. **Mandatory**: Yes, homebuyer education is mandatory for all HomeReady borrowers.
2. **Types of courses**: Acceptable education courses include pre-purchase homebuyer education and post-purchase housing counseling, both of which must be provided by an approved housing counseling agency. The content should cover topics such as budgeting, understanding mortgage payments, home maintenance, and homeownership responsibilities.
3. **Timing**: 
   - **Pre-purchase homebuyer education** must be completed **before the application is submitted**.
   - **Post-purchase housing counseling** must be completed **before closing**

In [20]:
# Test Case 3: Out-of-scope (should return 'cannot answer')
email_3 = """
Subject: Travel Recommendations for Business Trip

Hi there,

I'm planning a business trip to Seattle next month and was wondering if you could
recommend some good hotels near the downtown convention center.

Thanks,
Jennifer Martinez
"""
print("=== Test Case 3: Out-of-scope ===")
print(invoke_agent(SUPERVISOR_ARN, email_3))
# Expected: supervisor says it cannot answer this request

=== Test Case 3: Out-of-scope ===
<thinking> The email from Jennifer Martinez is asking for hotel recommendations, which does not fall under any of the categories that the specialist agents handle (System & Technology Issues, Policy & Guidelines Clarification, or Underwriting & Eligibility Questions). Therefore, the intent of this email is UNKNOWN, and I should respond accordingly. </thinking>

I'm sorry, but I'm unable to provide hotel recommendations or assist with travel-related inquiries. If you have any questions related to mortgage support, please feel free to ask.



## Bonus: Multi-turn Memory

The supervisor remembers context across turns
within the same session, thanks to AgentCore Memory.

In [21]:
import uuid
SESSION_ID = str(uuid.uuid4())
ACTOR_ID   = "student-demo"

# Turn 1 -- ask about a case
resp1 = agentcore_runtime.invoke_agent_runtime(
    agentRuntimeArn=SUPERVISOR_ARN,
    qualifier="DEFAULT",
    runtimeSessionId=SESSION_ID,
    payload=json.dumps({
        "prompt": "Can you look up case number 4567890123?",
        "actor_id": ACTOR_ID,
    }),
)
print("Turn 1:", json.loads(resp1["response"].read()).get("result"))

# Turn 2 -- follow-up in the same session -- agent remembers the case
resp2 = agentcore_runtime.invoke_agent_runtime(
    agentRuntimeArn=SUPERVISOR_ARN,
    qualifier="DEFAULT",
    runtimeSessionId=SESSION_ID,
    payload=json.dumps({
        "prompt": "What were the outstanding items on that case?",
        "actor_id": ACTOR_ID,
    }),
)
print("Turn 2:", json.loads(resp2["response"].read()).get("result"))

Turn 1: I'm unable to assist with your request. Please provide more details or specify the type of issue you're encountering.

Turn 2: I'm unable to retrieve the details for case number 4567890123 due to an authorization error. It seems that the required permissions are denied by a service control policy. 

To proceed, you may want to:
1. Verify that you have the appropriate permissions to access support case details.
2. Contact your system administrator to ensure your role has the necessary permissions.
3. Try accessing the case through your organization's support portal if available.

If you have any other policy questions or need guidance on mortgage selling guide topics, I'd be happy to help with that instead.

